# Survey Item Similarity Calculator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YetiKetil/Cosine_similarity_retriever/blob/main/SurveySimilarity.ipynb)

Compute pairwise **semantic similarity** between survey items (questionnaire statements), using your choice of method:

| Method | What it does | API key needed |
|---|---|---|
| **Local (offline)** | Free sentence-transformer model running inside Colab (`all-MiniLM-L6-v2`) | None |
| **OpenAI Small / Large** | `text-embedding-3-small` / `text-embedding-3-large` + cosine similarity | OpenAI |
| **Voyage AI** | `voyage-3` embeddings + cosine similarity | Voyage |
| **Claude (Semantic)** | Asks Claude to *rate* similarity of each pair (0.0–1.0) – not an embedding | Anthropic |

This is a Python port of the macOS/iOS SwiftUI app *SurveySimilarity*, so it runs on any computer with a browser.

### How to use
1. **Add your own API keys** (step 1 below). Never paste keys into a notebook you share.
2. Paste your survey items into the cell in step 2 (one per line, or paragraphs separated by a blank line).
3. Choose a method in step 3 and run all cells (*Runtime → Run all*).
4. View results as a list, a matrix and a heatmap, and download CSV files.

> **Costs:** OpenAI, Voyage and Anthropic charge per call (fractions of a cent for short items with embeddings). The Claude method makes one call per *pair* of items, i.e. n·(n−1)/2 calls – with 30 items that is 435 calls. Start with the free local method to test.

## 0. Setup

In [ ]:
import json, re, time, math, itertools
import numpy as np
import pandas as pd
import requests

try:
    from google.colab import userdata, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)

## 1. API keys

**Recommended (Colab):** click the 🔑 *Secrets* icon in the left sidebar and add any of these names, then switch on *Notebook access*:

- `OPENAI_API_KEY` – from https://platform.openai.com/api-keys
- `VOYAGE_API_KEY` – from https://www.voyageai.com/
- `ANTHROPIC_API_KEY` – from https://console.anthropic.com/

Secrets are stored in your own Google account and are **not** saved in the notebook or shared if you share it.
If a key is not found as a secret, the cell below asks you to paste it (hidden input) – but only for the method you actually use. Just press Enter to skip.

In [ ]:
import os
from getpass import getpass

def get_key(name):
    # 1) Colab secret, 2) environment variable, 3) hidden prompt
    if IN_COLAB:
        try:
            v = userdata.get(name)
            if v:
                return v.strip()
        except Exception:
            pass
    v = os.environ.get(name)
    if v:
        return v.strip()
    return getpass(f"{name} (press Enter to skip): ").strip()

KEYS = {}  # filled lazily, only when a method needs it

def key_for(name):
    if not KEYS.get(name):
        KEYS[name] = get_key(name)
    if not KEYS[name]:
        raise ValueError(f"No {name} provided. Add it under Colab Secrets (🔑) or paste it when asked.")
    return KEYS[name]

## 2. Enter your survey items

Paste items between the triple quotes. Either **one item per line**, or **paragraphs separated by an empty line** (same rule as the app: if splitting on blank lines gives fewer than 2 items, it splits on single lines).

*Alternative:* set `UPLOAD_FILE = True` to upload a `.csv`, `.xlsx` or `.txt` file instead (first column / first text column is used).

In [ ]:
ITEMS_TEXT = """
I enjoy my work.
My job gives me a sense of accomplishment.
I feel like quitting my job.
I often think about finding another job.
My supervisor treats me fairly.
"""

UPLOAD_FILE = False   # set to True to upload a file instead

In [ ]:
def parse_items(text):
    parts = [p.strip() for p in text.split("\n\n") if p.strip()]
    if len(parts) < 2:
        parts = [p.strip() for p in text.split("\n") if p.strip()]
    return parts

def items_from_upload():
    if not IN_COLAB:
        raise RuntimeError("File upload widget only works in Colab. Set ITEMS_TEXT instead.")
    up = colab_files.upload()
    name = next(iter(up))
    if name.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(name)
    elif name.lower().endswith(".csv"):
        df = pd.read_csv(name, sep=None, engine="python")
    else:
        return parse_items(up[name].decode("utf-8", errors="replace"))
    col = next((c for c in df.columns if df[c].dtype == object), df.columns[0])
    print(f"Using column: {col!r}")
    return [str(x).strip() for x in df[col].dropna() if str(x).strip()]

items = items_from_upload() if UPLOAD_FILE else parse_items(ITEMS_TEXT)
labels = [f"Item{i+1}" for i in range(len(items))]
assert len(items) >= 2, "Please enter at least 2 items."
print(f"{len(items)} items, {len(items)*(len(items)-1)//2} pairs\n")
for l, t in zip(labels, items):
    print(f"{l}: {t}")

## 3. Choose method

Options: `"local"`, `"openai_small"`, `"openai_large"`, `"voyage"`, `"claude"`

You can also list several, e.g. `METHODS = ["local", "openai_large", "voyage"]`, to compare methods side by side.

In [ ]:
METHODS = ["local"]

# Model names (change here if a provider renames/retires a model)
OPENAI_MODELS = {"openai_small": "text-embedding-3-small",
                 "openai_large": "text-embedding-3-large"}
VOYAGE_MODEL  = "voyage-3"
CLAUDE_MODEL  = "claude-sonnet-4-5"   # see https://docs.claude.com/en/docs/about-claude/models
LOCAL_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"

## 4. Embedding & similarity functions

In [ ]:
def cosine_matrix(E):
    E = np.asarray(E, dtype=float)
    n = np.linalg.norm(E, axis=1, keepdims=True)
    n[n == 0] = 1.0
    E = E / n
    return E @ E.T

def _post(url, headers, payload, retries=4):
    for attempt in range(retries):
        r = requests.post(url, headers=headers, json=payload, timeout=120)
        if r.status_code in (429, 500, 502, 503, 529) and attempt < retries - 1:
            time.sleep(2 ** attempt * 2)   # back off on rate limits / overload
            continue
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code} from {url}:\n{r.text[:500]}")
        return r.json()

def embed_openai(texts, model):
    key = key_for("OPENAI_API_KEY")
    out = []
    for i in range(0, len(texts), 256):          # batch for speed
        j = _post("https://api.openai.com/v1/embeddings",
                  {"Authorization": f"Bearer {key}"},
                  {"model": model, "input": texts[i:i+256]})
        out += [d["embedding"] for d in sorted(j["data"], key=lambda d: d["index"])]
    return np.array(out)

def embed_voyage(texts, model=VOYAGE_MODEL):
    key = key_for("VOYAGE_API_KEY")
    out = []
    for i in range(0, len(texts), 128):
        j = _post("https://api.voyageai.com/v1/embeddings",
                  {"Authorization": f"Bearer {key}"},
                  {"model": model, "input": texts[i:i+128]})
        out += [d["embedding"] for d in sorted(j["data"], key=lambda d: d["index"])]
    return np.array(out)

_local_model = None
def embed_local(texts):
    global _local_model
    if _local_model is None:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
            from sentence_transformers import SentenceTransformer
        _local_model = SentenceTransformer(LOCAL_MODEL)
    return _local_model.encode(texts)

CLAUDE_PROMPT = """Rate the semantic similarity between these two survey items on a scale from 0.0 to 1.0, where:
- 0.0 means completely unrelated
- 1.0 means identical in meaning
- Values in between represent degrees of semantic similarity

Item 1: "{a}"
Item 2: "{b}"

Respond with ONLY a decimal number between 0.0 and 1.0, nothing else."""

def claude_similarity(a, b, model=CLAUDE_MODEL):
    key = key_for("ANTHROPIC_API_KEY")
    j = _post("https://api.anthropic.com/v1/messages",
              {"x-api-key": key, "anthropic-version": "2023-06-01"},
              {"model": model, "max_tokens": 50,
               "messages": [{"role": "user", "content": CLAUDE_PROMPT.format(a=a, b=b)}]})
    text = "".join(c.get("text", "") for c in j.get("content", []))
    m = re.search(r"[-+]?\d*\.?\d+", text)
    if not m:
        raise RuntimeError(f"Could not parse a number from Claude's reply: {text!r}")
    return float(m.group())

def claude_matrix(texts):
    n = len(texts)
    S = np.eye(n)
    pairs = list(itertools.combinations(range(n), 2))
    for k, (i, j) in enumerate(pairs, 1):
        S[i, j] = S[j, i] = claude_similarity(texts[i], texts[j])
        print(f"\rClaude: {k}/{len(pairs)} pairs", end="")
    print()
    return S

METHOD_NAMES = {"local": "Local (all-MiniLM-L6-v2)",
                "openai_small": "OpenAI Small", "openai_large": "OpenAI Large",
                "voyage": "Voyage AI", "claude": "Claude (Semantic)"}

def similarity_matrix(method, texts):
    if method == "local":
        return cosine_matrix(embed_local(texts))
    if method in OPENAI_MODELS:
        return cosine_matrix(embed_openai(texts, OPENAI_MODELS[method]))
    if method == "voyage":
        return cosine_matrix(embed_voyage(texts))
    if method == "claude":
        return claude_matrix(texts)
    raise ValueError(f"Unknown method {method!r}. Choose from {list(METHOD_NAMES)}")

## 5. Run

In [ ]:
results = {}
for m in METHODS:
    print(f"Processing with {METHOD_NAMES[m]} ...")
    S = similarity_matrix(m, items)
    results[m] = pd.DataFrame(S, index=labels, columns=labels)
    print(f"✓ Calculated {len(items)*(len(items)-1)//2} pairwise similarities for {len(items)} items using {METHOD_NAMES[m]}.")

## 6. Results – list, matrix and heatmap

In [ ]:
def long_format(df, model_name):
    rows = [(df.index[i], df.columns[j], df.iat[i, j], model_name)
            for i, j in itertools.combinations(range(len(df)), 2)]
    return pd.DataFrame(rows, columns=["Item1", "Item2", "Similarity", "Model"])

try:
    from IPython.display import display
except ImportError:
    display = print

# Coloured tables need jinja2 (pre-installed in Colab). Try to install it if missing;
# otherwise fall back to plain tables so the notebook never stops here.
try:
    import jinja2
    HAS_STYLE = True
except ImportError:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jinja2"], check=True)
        import jinja2
        HAS_STYLE = True
    except Exception:
        HAS_STYLE = False
        print("Note: jinja2 not available – showing tables without colours.")

def colour(v):
    if not isinstance(v, (int, float)): return ""
    c = "#c8f7c5" if v > 0.7 else "#ffe0b3" if v > 0.4 else "#f9c6c6"
    return f"background-color: {c}; color: black"

def show(df, cols=None, hide_index=False):
    # Coloured table if possible, plain rounded table otherwise
    if HAS_STYLE:
        try:
            st = df.style.format("{:.3f}", subset=cols)
            st = st.map(colour, subset=cols) if hasattr(st, "map") else st.applymap(colour, subset=cols)
            if hide_index:
                st = st.hide(axis="index")
            display(st)
            return
        except Exception:
            pass
    display(df.round(3) if cols is None else df.round({c: 3 for c in cols}))

for m, df in results.items():
    print(f"\n=== {METHOD_NAMES[m]} ===")
    lf = long_format(df, METHOD_NAMES[m]).sort_values("Similarity", ascending=False)
    show(lf, cols=["Similarity"], hide_index=True)
    if len(items) <= 30:
        show(df)

In [ ]:
import matplotlib.pyplot as plt

for m, df in results.items():
    n = len(df)
    fig, ax = plt.subplots(figsize=(max(5, 0.5*n + 2), max(4, 0.5*n + 1.5)))
    im = ax.imshow(df.values, cmap="viridis", vmin=min(0, df.values.min()), vmax=1)
    ax.set_xticks(range(n), df.columns, rotation=90); ax.set_yticks(range(n), df.index)
    if n <= 20:
        for i in range(n):
            for j in range(n):
                ax.text(j, i, f"{df.iat[i,j]:.2f}", ha="center", va="center",
                        color="white" if df.iat[i,j] < 0.6 else "black", fontsize=8)
    ax.set_title(METHOD_NAMES[m]); fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout(); plt.show()

### Comparing methods (only if you ran more than one)
Correlation between the pairwise similarities produced by each method.

In [ ]:
if len(results) > 1:
    wide = pd.DataFrame({METHOD_NAMES[m]: long_format(df, m)["Similarity"].values
                         for m, df in results.items()})
    display(wide.corr().round(3))
else:
    print("Only one method run – add more to METHODS to compare.")

## 7. Export CSV

Writes, for each method, a long file (`Item1, Item2, Similarity, Model` – same format as the app) and a full matrix file, plus a key file with item texts.

In [ ]:
written = []
pd.DataFrame({"Label": labels, "Item": items}).to_csv("survey_items_key.csv", index=False)
written.append("survey_items_key.csv")
for m, df in results.items():
    tag = METHOD_NAMES[m].split(" (")[0].replace(" ", "_")
    f1, f2 = f"survey_similarities_{tag}.csv", f"survey_similarity_matrix_{tag}.csv"
    long_format(df, METHOD_NAMES[m]).round(3).to_csv(f1, index=False)
    df.round(3).to_csv(f2)
    written += [f1, f2]
print("Wrote:", *written, sep="\n  ")
if IN_COLAB:
    for f in written:
        colab_files.download(f)

---
**Notes**
- Cosine similarity values from different embedding models are not on the same scale – e.g. OpenAI models tend to give higher baseline values than local models. Compare *patterns* within a model, or correlations across models, rather than raw numbers across models.
- The Claude method is a rating by a language model, not a cosine similarity; it can vary slightly between runs.
- If a provider retires a model name, change it in step 3.